# Deployment / meta-test  (Algorithm 10.3)

Everything is **frozen** here — the context encoder, the perturbation encoder, all
policies/critics, and the change-point detector. No gradient updates. A dedicated
single-environment test instance is driven through **scripted non-stationarity
events**:

* `RegimeChangeEvent(step, agent, regime)` — an abrupt dynamics change for one agent
  (Section 10.1's $\mu_i \to \mu_{2,i}$); what the posterior + detector must catch.
* `AdversarialAttackEvent(step, agent)` — activates the position + own-action PGD
  attacks against one agent from `step` onward.

We look at four scenarios (**nominal**, **regime change**, **adversarial attack**,
**both at once**) and report, per scenario: victims rescued, return, **detection
(fired? delay)**, **recovery time**, and **robustness degradation** vs nominal.

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if (ROOT / "method").exists(): sys.path.insert(0, str(ROOT))
from method.io import load_model
from method.training.meta_test import MetaTestRunner, MetaTestConfig, RegimeChangeEvent, AdversarialAttackEvent
from method.training.regime import Regime
from method.viz import _victim_state, animate
from method import metrics as M
from utils.scenario import Scenario

# ---- what is under test (point these at any checkpoint / detector) ----
RUN       = ROOT / "runs" / "retrain_v3_phys"
P1_CKPT   = RUN / "phase1_checkpoint_14999.pt"
DETECTOR  = RUN / "phase2_detector.pt"
CFG       = "world_config_5v.yaml"
CHANGE_STEP  = 60
MAX_STEPS    = 220
THRESHOLD_C        = 0.5     # detector fires when p >= THRESHOLD_C ...
TRIGGER_PERSIST    = 3       # ... for this many consecutive steps

# regime pair for the "change" scenario: two DEFAULT_MODES centroids
NORMAL, HEAVY = Regime(1.10, 0.16), Regime(1.52, 0.36)
plt.rcParams["figure.dpi"] = 110

trainer, detector = load_model(str(P1_CKPT), str(DETECTOR))
NA = trainer.n_agents
print("under test:", P1_CKPT.name, "+", DETECTOR.name)
print(f"n_agents={NA}  config={CFG}  detector gate: p>= {THRESHOLD_C} for {TRIGGER_PERSIST} steps")

under test: phase1_checkpoint_14999.pt + phase2_detector.pt
n_agents=2  config=world_config_5v.yaml  detector gate: p>= 0.5 for 3 steps


In [2]:
def deploy_episode(kind, seed, render_every=0, threshold_C=THRESHOLD_C, persist=TRIGGER_PERSIST,
                   max_steps=MAX_STEPS):
    '''kind in {nominal, change, attack, both}. Returns a dict of per-step traces.'''
    torch.manual_seed(seed)
    ev = [RegimeChangeEvent(0, i, NORMAL) for i in range(NA)]
    tc = None
    if kind in ("change", "both"):
        ev.append(RegimeChangeEvent(CHANGE_STEP, 0, HEAVY)); tc = CHANGE_STEP
    if kind in ("attack", "both"):
        ev.append(AdversarialAttackEvent(CHANGE_STEP, 0, True, True))
    r = MetaTestRunner(trainer, detector,
                       scenario_factory=lambda: Scenario(config_file=CFG, fit_map=True),
                       config=MetaTestConfig(threshold_C=threshold_C, trigger_persistence=persist))
    r.env.scenario.viewer_zoom = 2.65
    r.schedule(ev); r.reset_state()
    P, RS, REW, MODE, HP, ATK, SIG, frames, metas = [], [], [], [], [], [], [], [], []
    for t in range(max_steps):
        s = r.step(render=(render_every and t % render_every == 0))
        P.append(float(s.detector_p[0] or 0.0))
        REW.append(float(np.sum(s.rewards)))
        MODE.append(s.mode[0]); ATK.append(bool(s.active_attacks[0]))
        SIG.append(float(r.posterior_state.sigma2[0, 0]))
        if s.reset_fired[0]: RS.append(t)
        if s.frame is not None: frames.append(np.asarray(s.frame)); metas.append(s)
        h, _, done = _victim_state(r.env)
        HP.append(h)
        if done: break
    scen = r.env.scenario
    rescued = int(sum(bool(sv.rescued[0]) for sv in scen._survivals))
    return dict(kind=kind, tc=tc, seed=seed, p=np.array(P), resets=RS, step_rewards=np.array(REW),
                mode=MODE, min_health=np.array(HP), attacked=np.array(ATK), sigma2=np.array(SIG),
                steps=len(P), total_return=float(np.sum(REW)), rescued=rescued,
                frames=frames, metas=metas)

def mode_bands(ax, mode):
    ex = np.array([m == "execute" for m in mode])
    ax.fill_between(range(len(mode)), 0, 1, where=ex, transform=ax.get_xaxis_transform(),
                    color="C2", alpha=.08, label="agent-0 EXECUTE")
    ax.fill_between(range(len(mode)), 0, 1, where=~ex, transform=ax.get_xaxis_transform(),
                    color="C1", alpha=.08, label="agent-0 EXPLORE")
print("helper ready")

helper ready


## Scenario A — nominal deployment (no events)

In [3]:
A = deploy_episode("nominal", seed=100, render_every=25)
print({k: A[k] for k in ["steps","rescued","total_return"]})
fig, ax = plt.subplots(2, 1, figsize=(11, 4.6), sharex=True)
mode_bands(ax[0], A["mode"])
ax[0].plot(A["min_health"], color="C3"); ax[0].set_ylabel("min victim\nhealth"); ax[0].legend(fontsize=7, loc="lower left")
ax[0].set_title(f"Scenario A: nominal  (rescued {A['rescued']}/5 in {A['steps']} steps, return {A['total_return']:.1f})")
ax[1].plot(np.cumsum(A["step_rewards"]), color="C0"); ax[1].set_ylabel("cumulative\nreturn"); ax[1].set_xlabel("step")
fig.tight_layout(); plt.show()
if A["frames"]:
    n=len(A["frames"]); fig,axs=plt.subplots(1,n,figsize=(2.6*n,2.6))
    for a,fr,mt in zip(np.atleast_1d(axs), A["frames"], A["metas"]):
        a.imshow(fr); a.axis("off"); a.set_title(f"t={mt.step}", fontsize=8)
    fig.tight_layout(); plt.show()

{'steps': 220, 'rescued': 2, 'total_return': -7.239677295088768}


## Scenario B — abrupt regime change (agent 0, normal → heavy @ t = 60)

In [4]:
B = deploy_episode("change", seed=200)
tc = B["tc"]
# detection: first reset within 10 steps of the true change
post = [k for k in B["resets"] if tc <= k <= tc + 10]
delay = (post[0] - tc) if post else np.nan
rt, cens = M.recovery_time(B["step_rewards"], tc, B["steps"])

fig, ax = plt.subplots(3, 1, figsize=(11, 7.5), sharex=True)
a2 = ax[0].twinx(); a2.plot(B["sigma2"], color="0.6", lw=1); a2.set_ylabel(r"posterior $\sigma^2$ (agent 0)", color="0.5", fontsize=8)
ax[0].plot(B["p"], color="C0", lw=1.4, label="detector $p_t$")
ax[0].axhline(THRESHOLD_C, ls="--", color="k", lw=.7); ax[0].axvline(tc, color="C3", lw=2, label=f"true change (t={tc})")
for k in B["resets"]: ax[0].axvline(k, color="C2", ls=":", lw=1.6)
if post: ax[0].text(post[0]+1, .5, f"detected, delay {delay}", color="C2", fontsize=9)
else:    ax[0].text(tc+12, .5, "no detection", color="C3", fontsize=9)
ax[0].set_ylabel("$p_t$"); ax[0].set_ylim(-.05,1.05); ax[0].legend(fontsize=7, loc="upper left")
ax[0].set_title("Scenario B: regime change — detector, posterior reset (sawtooth)")

mode_bands(ax[1], B["mode"]); ax[1].plot(B["min_health"], color="C3")
ax[1].axvline(tc, color="C3", lw=2); ax[1].set_ylabel("min victim\nhealth"); ax[1].legend(fontsize=7, loc="lower left")

r = B["step_rewards"]; ax[2].plot(r, color="C0", lw=.8, alpha=.5)
ax[2].plot(np.convolve(r, np.ones(20)/20, "same"), color="C0", lw=1.6, label="20-step MA")
if tc >= 20:
    ref = r[tc-20:tc].mean(); ax[2].axhspan(ref*0.9, ref*1.1, color="C2", alpha=.15, label="±10% pre-change band")
ax[2].axvline(tc, color="C3", lw=2); ax[2].set_ylabel("team reward\n/ step"); ax[2].set_xlabel("step"); ax[2].legend(fontsize=7)
fig.tight_layout(); plt.show()

print(f"detection delay : {delay}  (nan = missed)")
print(f"recovery time   : {rt:.0f} steps{'  (censored — never recovered in-episode)' if cens else ''}")
print(f"rescued {B['rescued']}/5,  return {B['total_return']:.1f}  ({B['steps']} steps)")

detection delay : nan  (nan = missed)
recovery time   : 0 steps
rescued 0/5,  return -44.0  (220 steps)


## Scenario C — adversarial attack (agent 0, position + action PGD @ t = 60)

In [5]:
C_ = deploy_episode("attack", seed=300)
A2 = deploy_episode("nominal", seed=300)   # matched seed, no attack
deg = M.degradation(A2["total_return"], C_["total_return"])

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(np.cumsum(A2["step_rewards"]), label="nominal", color="C2")
ax[0].plot(np.cumsum(C_["step_rewards"]), label="under attack", color="C3")
ax[0].axvline(CHANGE_STEP, color="k", ls="--", lw=1, label="attack ON")
ax[0].set_title("cumulative return: nominal vs attacked"); ax[0].set_xlabel("step"); ax[0].legend(fontsize=8)
ax[1].bar(["nominal","attacked"], [A2["total_return"], C_["total_return"]], color=["C2","C3"])
ax[1].set_title(f"episode return  (degradation: {deg['degradation_abs']:.1f} abs, "
                f"{100*deg['degradation_rel']:.0f}% rel)")
fig.tight_layout(); plt.show()
print(f"nominal return {A2['total_return']:.1f} (rescued {A2['rescued']}/5)  ->  "
      f"attacked {C_['total_return']:.1f} (rescued {C_['rescued']}/5)")
print("degradation:", {k: round(v,3) for k,v in deg.items()})

nominal return -25.4 (rescued 1/5)  ->  attacked -44.0 (rescued 0/5)
degradation: {'degradation_abs': 18.62, 'degradation_rel': 0.734}


## Scenario D — regime change **and** attack, simultaneously @ t = 60

In [6]:
D = deploy_episode("both", seed=400)
postD = [k for k in D["resets"] if CHANGE_STEP <= k <= CHANGE_STEP+10]
rtD, censD = M.recovery_time(D["step_rewards"], CHANGE_STEP, D["steps"])
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax[0].plot(D["p"], color="C0", label="detector $p_t$"); ax[0].axhline(THRESHOLD_C, ls="--", color="k", lw=.7)
ax[0].axvline(CHANGE_STEP, color="C3", lw=2, label="change + attack ON")
for k in D["resets"]: ax[0].axvline(k, color="C2", ls=":", lw=1.6)
ax[0].fill_between(range(D["steps"]), 0, 1, where=D["attacked"], transform=ax[0].get_xaxis_transform(),
                   color="C3", alpha=.08, label="agent-0 attacked")
ax[0].set_ylabel("$p_t$"); ax[0].set_ylim(-.05,1.05); ax[0].legend(fontsize=7, loc="upper left")
mode_bands(ax[1], D["mode"]); ax[1].plot(np.cumsum(D["step_rewards"]), color="C0")
ax[1].axvline(CHANGE_STEP, color="C3", lw=2); ax[1].set_ylabel("cumulative\nreturn"); ax[1].set_xlabel("step")
ax[0].set_title("Scenario D: simultaneous change + attack"); fig.tight_layout(); plt.show()
print(f"detection delay {(postD[0]-CHANGE_STEP) if postD else float('nan')}  |  "
      f"recovery {rtD:.0f}{' (censored)' if censD else ''}  |  rescued {D['rescued']}/5  return {D['total_return']:.1f}")

detection delay nan  |  recovery 0  |  rescued 0/5  return -44.0


## Deployment metrics — summary table

In [7]:
def row(name, e, base_return):
    tc = e["tc"] if e["tc"] is not None else CHANGE_STEP
    trig = [k for k in e["resets"] if tc <= k <= tc+10]
    rt, cens = (M.recovery_time(e["step_rewards"], e["tc"], e["steps"]) if e["tc"] is not None else (np.nan, False))
    d = M.degradation(base_return, e["total_return"])
    return dict(scenario=name, rescued=f"{e['rescued']}/5", steps=e["steps"],
                return_=round(e["total_return"],1),
                detection=("fired" if trig else ("—" if e["tc"] is None else "MISSED")),
                delay=(trig[0]-tc if trig else np.nan),
                n_resets=len(e["resets"]),
                recovery=("" if e["tc"] is None else (f"{rt:.0f}{'*' if cens else ''}")),
                degr_vs_nominal_pct=round(100*d["degradation_rel"],0))
base = A["total_return"]
tbl = pd.DataFrame([row("A nominal", A, base), row("B change", B, base),
                    row("C attack", C_, base), row("D both", D, base)])
display(tbl)
print("delay in steps (nan = missed); recovery in steps ('*' = censored, never recovered); "
      "degr_vs_nominal_pct relative to Scenario A return.")

,scenario,rescued,steps,return_,detection,delay,n_resets,recovery,degr_vs_nominal_pct
0,A nominal,2/5,220,-7.2,—,NaN,0,,0.0
1,B change,0/5,220,-44.0,MISSED,NaN,0,0,508.0
2,C attack,0/5,220,-44.0,—,NaN,0,,508.0
3,D both,0/5,220,-44.0,MISSED,NaN,0,0,508.0


delay in steps (nan = missed); recovery in steps ('*' = censored, never recovered); degr_vs_nominal_pct relative to Scenario A return.


## Interactive playback — scrub one episode

In [8]:
from IPython.display import HTML
PB = deploy_episode("change", seed=222, render_every=1)
ev_by_step = {CHANGE_STEP: ["regime change: agent 0"]}
extra = [{"health": h, "rescued": False, "done": False} for h in PB["min_health"]]
anim = animate(PB["frames"], PB["metas"], extra, ev_by_step,
               title="deployment: agent-0 regime change @ 60", fps=6)
HTML(anim.to_jshtml())

## Operating-point sweep — detector threshold × persistence

In [9]:
NS = 10
chg = [deploy_episode("change", 1000+i) for i in range(NS)]
noc = [deploy_episode("nominal", 5000+i) for i in range(NS)]
def redo(e, thr, persist):
    return [k for k in range(len(e["p"])) if k+1>=persist and np.all(e["p"][k+1-persist:k+1] >= thr)]
rows = []
for thr in [0.3, 0.5, 0.7]:
    for persist in [1, 3, 5]:
        TP=FN=0; delays=[]
        for e in chg:
            trig=redo(e, thr, persist); post=[k for k in trig if CHANGE_STEP<=k<=CHANGE_STEP+10]
            if post: TP+=1; delays.append(post[0]-CHANGE_STEP)
            else: FN+=1
        TN=sum(1 for e in noc if not redo(e, thr, persist)); FP=NS-TN
        rows.append(dict(threshold=thr, persistence=persist,
                         accuracy=round((TP+TN)/(2*NS),2), TPR=round(TP/NS,2), TNR=round(TN/NS,2),
                         FA_rate=round(FP/NS,2),
                         delay_median=(np.median(delays) if delays else np.nan)))
display(pd.DataFrame(rows))

,threshold,persistence,accuracy,TPR,TNR,FA_rate,delay_median
0,0.3,1,0.5,0.0,1.0,0.0,NaN
1,0.3,3,0.5,0.0,1.0,0.0,NaN
2,0.3,5,0.5,0.0,1.0,0.0,NaN
3,0.5,1,0.5,0.0,1.0,0.0,NaN
4,0.5,3,0.5,0.0,1.0,0.0,NaN
5,0.5,5,0.5,0.0,1.0,0.0,NaN
6,0.7,1,0.5,0.0,1.0,0.0,NaN
7,0.7,3,0.5,0.0,1.0,0.0,NaN
8,0.7,5,0.5,0.0,1.0,0.0,NaN


## Findings / how to use this notebook

The **deployment procedure** is exercised end-to-end: frozen model, scripted regime
change and adversarial attack on one agent, per-step detector output, posterior
reset → re-explore → re-execute, recovery, and robustness degradation.

**Current checkpoint (`retrain_v3_phys`) caveat:** the change-point detector is
degenerate — `p_t ≈ 0` on every step — so Scenarios **B** and **D** show *no
detection*, detection delay is undefined, and the posterior is never reset by the
detector (the sawtooth in B is flat). Scenario **C** (attack) and the reward /
recovery numbers are meaningful; the detection rows are not.

When a working detector is available (the BCE-anchor Phase-2 retrain), point
`P1_CKPT` / `DETECTOR` at it and re-run — every plot and the summary table update
automatically.